# Feature Track 3: Context Enrichment via Neighbouring Chunks

---

The retrieval strategies in `feature3_advanced_retrieval.ipynb` address **which chunks** to
retrieve. This notebook addresses **how much context** to give the LLM once those chunks are found.

### The problem: chunks that reference their neighbours

Header-based chunking (the default) splits at section boundaries, so each chunk is
naturally self-contained. Fixed-size and paragraph-aware chunking cut by character count,
which can split mid-section:

```
Chunk 3  │ "...base year 2019. The transport stage alone accounts for"
Chunk 4  │ "35% of this figure. Scope 3 emissions are not included."
```

If the retriever finds chunk 4, the LLM sees `"35% of this figure"` with no antecedent.
The answer will be incomplete or wrong.

### The fix: ContextWindowRetriever

```
Base retriever  ──► chunk 4  ──► ContextWindowRetriever ──► chunk 3 + chunk 4 + chunk 5
                                       (window_size=1)              (stitched content)
```

After retrieval, `ContextWindowRetriever` fetches the adjacent chunks (`chunk_index ± window_size`
within the same `source_file`) and stitches their content before the result reaches the LLM.

### Requirements

- Chunks must have `chunk_index` (int) and `source_file` (str) in their metadata.
- These are set by **`fixed_size_chunks()`** and **`paragraph_aware_chunks()`** from
  `feature0_ingestion.py`, but **not** by the default header-based `PDFChunker`.
- This notebook builds a separate vector store from fixed-size chunks.

---

## Setup

**Prerequisites:** `conversational-toolkit` and `backend` must be installed in editable mode.
For **OpenAI**, set `OPENAI_API_KEY`. For **Ollama**, start `ollama serve` and pull the model.

In [ ]:
from dataclasses import dataclass
from typing import Sequence

from conversational_toolkit.chunking.base import Chunk
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings

from conversational_toolkit.retriever.context_window_retriever import (
    ContextWindowRetriever,
)
from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever
from conversational_toolkit.vectorstores.base import ChunkRecord
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    DATA_DIR,
    EMBEDDING_MODEL,
    RETRIEVER_TOP_K,
    VS_PATH,
    build_llm,
    build_vector_store,
)
from sme_kt_zh_collaboration_rag.feature0_ingestion import fixed_size_chunks

# ---------------------------------------------------------------------------
# Fixed-size vector store path (separate from the default header-based store)
# ---------------------------------------------------------------------------

VS_FIXED_PATH = VS_PATH.parent / "data_vs_fixed.db"


# ---------------------------------------------------------------------------
# Data structures and helpers
# ---------------------------------------------------------------------------


@dataclass
class EnrichmentResult:
    strategy: str
    query: str
    chunks: Sequence[ChunkRecord]
    window_size: int = 0

    def __str__(self) -> str:
        lines = [f"[{self.strategy}  window={self.window_size}]"]
        for c in self.chunks:
            src = c.metadata.get("source_file", "?")
            idx = c.metadata.get("chunk_index", "?")
            lines.append(f"  chunk_index={idx:<4}  {src:<48}  {len(c.content)} chars")
        return "\n".join(lines)


def load_fixed_size_chunks(
    data_dir=DATA_DIR,
    chunk_size: int = 800,
    overlap: int = 100,
    max_files=None,
) -> list[Chunk]:
    """Load all PDFs from data_dir using fixed-size chunking with chunk_index metadata."""
    all_chunks: list[Chunk] = []
    files = sorted(f for f in data_dir.iterdir() if f.suffix.lower() == ".pdf")
    if max_files is not None:
        files = files[:max_files]
    for file_path in files:
        try:
            chunks = fixed_size_chunks(
                str(file_path), chunk_size=chunk_size, overlap=overlap
            )
            for c in chunks:
                c.metadata["source_file"] = file_path.name
            all_chunks.extend(chunks)
        except Exception as exc:
            print(f"Skipping {file_path.name}: {exc}")
    print(f"Loaded {len(all_chunks)} fixed-size chunks from {len(files)} files")
    return all_chunks


async def build_fixed_size_vector_store(
    embedding_model,
    chunk_size: int = 800,
    overlap: int = 100,
    db_path=VS_FIXED_PATH,
    reset: bool = False,
) -> ChromaDBVectorStore:
    """Build (or reuse) a ChromaDB vector store from fixed-size chunks."""
    chunks = load_fixed_size_chunks(chunk_size=chunk_size, overlap=overlap)
    return await build_vector_store(
        chunks, embedding_model, db_path=db_path, reset=reset
    )


async def retrieve_baseline(
    query: str,
    embedding_model,
    vector_store: ChromaDBVectorStore,
    top_k: int = 5,
) -> EnrichmentResult:
    """Baseline: retrieve top-k chunks without any context expansion."""
    retriever = VectorStoreRetriever(embedding_model, vector_store, top_k=top_k)
    chunks = await retriever.retrieve(query)
    return EnrichmentResult("baseline", query, chunks, window_size=0)


async def retrieve_with_context_window(
    query: str,
    embedding_model,
    vector_store: ChromaDBVectorStore,
    top_k: int = 5,
    window_size: int = 1,
) -> EnrichmentResult:
    """Context-enriched retrieval: each retrieved chunk is expanded with its neighbours."""
    base = VectorStoreRetriever(embedding_model, vector_store, top_k=top_k)
    retriever = ContextWindowRetriever(
        retriever=base, vector_store=vector_store, window_size=window_size, top_k=top_k
    )
    chunks = await retriever.retrieve(query)
    return EnrichmentResult("context_window", query, chunks, window_size=window_size)


def print_enrichment_comparison(
    baseline: EnrichmentResult,
    enriched: EnrichmentResult,
    relevant_keywords: list[str],
    show_content_chars: int = 300,
) -> None:
    """Print a before/after comparison of baseline vs. context-enriched retrieval."""

    def _keyword_hit(content: str) -> bool:
        return any(kw.lower() in content.lower() for kw in relevant_keywords)

    print(f"Query: {baseline.query!r}")
    print(f"Keywords to find: {relevant_keywords}\n")

    for i, (b_chunk, e_chunk) in enumerate(zip(baseline.chunks, enriched.chunks), 1):
        b_hit = _keyword_hit(b_chunk.content)
        e_hit = _keyword_hit(e_chunk.content)
        gained = e_hit and not b_hit
        marker = "🆕" if gained else ("✓" if e_hit else "·")

        src = b_chunk.metadata.get("source_file", "?")
        idx = b_chunk.metadata.get("chunk_index", "?")
        print(f"  [{i}] {marker}  chunk_index={idx}  {src}")
        print(
            f"       baseline  : {len(b_chunk.content):>5} chars  "
            f"keywords={'✓' if b_hit else '·'}"
        )
        print(
            f"       enriched  : {len(e_chunk.content):>5} chars  "
            f"keywords={'✓' if e_hit else '·'}"
            + ("  ← keyword gained by context!" if gained else "")
        )
        if show_content_chars > 0:
            print(f"       content   : {e_chunk.content[:show_content_chars]!r}")
        print()


# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------

# Choose your LLM backend
BACKEND = "openai"  # "ollama", "openai", or "qwen"

embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)
llm = build_llm(backend=BACKEND)

print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"LLM backend     : {BACKEND}")
print("Setup complete.")

---

## 1. Fixed-Size Chunks vs. Header-Based Chunks

The key difference between the two chunking strategies:

| Property | Header-based (default) | Fixed-size |
|---|---|---|
| Boundary | Markdown headings | Character count (800 chars, 100 overlap) |
| Size | Variable (depends on section length) | Predictable (~800 chars) |
| Self-contained? | Usually yes | Not guaranteed |
| `chunk_index` in metadata? | **No** | **Yes** |
| `source_file` in metadata? | Yes (added by `load_chunks`) | Yes (added by `load_fixed_size_chunks`) |

For context window expansion to work, we need `chunk_index`. We build a separate
vector store from fixed-size chunks.

In [ ]:
# Inspect the fixed-size chunks before building the vector store
chunks = load_fixed_size_chunks(chunk_size=800, overlap=100)

print(f"Total fixed-size chunks: {len(chunks)}")
print()
print("First 3 chunks from the same file (notice sequential chunk_index):")
first_file = chunks[0].metadata.get("source_file", "?")
same_file_chunks = [c for c in chunks if c.metadata.get("source_file") == first_file][
    :3
]
for c in same_file_chunks:
    idx = c.metadata.get("chunk_index", "?")
    src = c.metadata.get("source_file", "?")
    start = c.metadata.get("start_char", "?")
    end = c.metadata.get("end_char", "?")
    print(f"  chunk_index={idx:<4}  chars={start}-{end}  file={src}")
    print(f"  content: {c.content[:120].strip()!r}")
    print()

In [ ]:
# Find a pair of consecutive chunks where the second one references the first
# (sentence starts mid-thought, refers to 'this', 'above', 'the figure', etc.)
cross_boundary_clues = [
    "this figure",
    "as described",
    "above",
    "as noted",
    "therefore",
    "consequently",
    "these values",
    "as mentioned",
]

found = False
for i in range(len(chunks) - 1):
    a, b = chunks[i], chunks[i + 1]
    if a.metadata.get("source_file") != b.metadata.get("source_file"):
        continue
    b_start = b.content[:150].lower()
    if any(clue in b_start for clue in cross_boundary_clues):
        print(f"Found cross-boundary reference! file={b.metadata['source_file']}")
        print(f"  Chunk {a.metadata['chunk_index']}: ...{a.content[-150:].strip()!r}")
        print(f"  Chunk {b.metadata['chunk_index']}: {b.content[:200].strip()!r}")
        found = True
        break

if not found:
    print("No cross-boundary reference found with simple heuristics.")
    print("Try inspecting consecutive chunks manually:")
    a, b = same_file_chunks[0], same_file_chunks[1]
    print(
        f"  Chunk {a.metadata['chunk_index']} ends  : ...{a.content[-120:].strip()!r}"
    )
    print(f"  Chunk {b.metadata['chunk_index']} starts: {b.content[:120].strip()!r}")

---

## 2. Build the Fixed-Size Vector Store

This vector store is separate from the header-based one used in `feature0_baseline_rag.ipynb`.
It stores the same documents but chunked with fixed-size windows,
which preserves `chunk_index` in metadata, required for neighbour lookup.

In [ ]:
print(f"Building fixed-size vector store at: {VS_FIXED_PATH}")
print("(~30 s on first run; reused on subsequent runs)")

vector_store = await build_fixed_size_vector_store(
    embedding_model,
    chunk_size=800,
    overlap=100,
    reset=False,  # set True to force rebuild
)

count = vector_store.collection.count()
print(f"Vector store ready: {count} chunks")

---

## 3. Baseline Retrieval: Without Context Expansion

First, retrieve with the standard `VectorStoreRetriever` to establish the baseline.
Notice the character count of each chunk: these are the raw fixed-size chunks
that the LLM would receive without any expansion.

In [ ]:
QUERY = "What is the carbon footprint of the product?"

baseline_result = await retrieve_baseline(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K
)

print(f"[baseline]  query: {QUERY!r}\n")
for chunk in baseline_result.chunks[:RETRIEVER_TOP_K]:
    src = chunk.metadata.get("source_file", "?")
    idx = chunk.metadata.get("chunk_index", "?")
    print(
        f"  chunk_index={idx:<4}  {len(chunk.content):>4} chars  score={chunk.score:.4f}  {src}"
    )
    print(f"  content: {chunk.content[:150].strip()!r}")
    print()

---

## 4. Context-Enriched Retrieval: With Neighbour Expansion

Now wrap the same base retriever with `ContextWindowRetriever`.

For each retrieved chunk at `chunk_index = N`, the retriever fetches:
- Chunk at `chunk_index = N - window_size` … `N - 1` (predecessors)
- Chunk at `chunk_index = N + 1` … `N + window_size` (successors)

all from the **same `source_file`**. The content is stitched in order before returning.

**How the lookup works:**

```python
# For each offset in [-window_size, ..., -1, +1, ..., +window_size]:
neighbours = await vector_store.get_chunks_by_filter({
    "$and": [
        {"source_file": {"$eq": source}},
        {"chunk_index": {"$eq": idx + offset}},
    ]
})
```

This uses `get_chunks_by_filter()`: a metadata-only lookup with no embedding needed.

In [ ]:
enriched_result = await retrieve_with_context_window(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K, window_size=1
)

print(f"[context_window w=1]  query: {QUERY!r}\n")
for chunk in enriched_result.chunks[:RETRIEVER_TOP_K]:
    src = chunk.metadata.get("source_file", "?")
    idx = chunk.metadata.get("chunk_index", "?")
    print(
        f"  chunk_index={idx:<4}  {len(chunk.content):>5} chars  score={chunk.score:.4f}  {src}"
    )
    print(f"  content: {chunk.content[:200].strip()!r}")
    print()

> **Observation:** Each chunk is now larger: it includes the content of its neighbours.
The character count roughly triples for `window_size=1` (one chunk before + one after).
The LLM now has the antecedent context it needs to produce a complete answer.

---

## 5. Before / After Comparison

`print_enrichment_comparison()` highlights chunks where context expansion
"gained" a relevant keyword that was absent in the original chunk.
This is the key signal: the LLM can now answer questions that the baseline
retrieval would have answered incompletely.

In [ ]:
RELEVANT_KEYWORDS = [
    "CO₂",
    "carbon",
    "GWP",
    "footprint",
    "kg",
    "reduction",
    "transport",
]

print_enrichment_comparison(
    baseline_result,
    enriched_result,
    relevant_keywords=RELEVANT_KEYWORDS,
    show_content_chars=200,
)

In [ ]:
# Quantify the content growth
print("Content size comparison:\n")
print(f"{'Chunk':>5}  {'Baseline':>10}  {'Window=1':>10}  {'Window=2':>10}")
print("─" * 50)

enriched_w2 = await retrieve_with_context_window(
    QUERY, embedding_model, vector_store, top_k=RETRIEVER_TOP_K, window_size=2
)

for i, (b, e1, e2) in enumerate(
    zip(baseline_result.chunks, enriched_result.chunks, enriched_w2.chunks), 1
):
    print(f"{i:>5}  {len(b.content):>10}  {len(e1.content):>10}  {len(e2.content):>10}")

print()
print(
    "Note: window_size=2 fetches 2 chunks on each side — up to 5x the original content."
)
print("Be mindful of the LLM context window: larger windows consume more tokens.")

---

## 6. Token Budget Considerations

Context expansion increases content size. The LLM context window has a limit:

| Model | Context window |
|---|---|
| `gpt-4o-mini` | 128 000 tokens |
| `llama3.2:3b` (Ollama) | 128 000 tokens |
| `mistral-nemo:12b` (Ollama) | 128 000 tokens |
| `all-MiniLM-L6-v2` (embedding) | 256 tokens - affects **input**, not LLM |

For the current corpus and `RETRIEVER_TOP_K=5` with `window_size=1`:
- Baseline: ~5 × 800 chars ≈ 5 × 200 tokens ≈ **1 000 tokens**
- Window=1: ~5 × 2 400 chars ≈ 5 × 600 tokens ≈ **3 000 tokens**
- Window=2: ~5 × 4 000 chars ≈ 5 × 1 000 tokens ≈ **5 000 tokens**

All well within current model context windows. For very large corpora or longer chunks,
watch for truncation.

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough estimate: 1 token ≈ 4 characters."""
    return len(text) // 4


print("Estimated token usage per strategy:\n")
for label, result in [
    ("baseline   (w=0)", baseline_result),
    ("enriched   (w=1)", enriched_result),
    ("enriched   (w=2)", enriched_w2),
]:
    total_chars = sum(len(c.content) for c in result.chunks)
    total_tokens = estimate_tokens(total_chars)
    print(f"  {label:<22}  {total_chars:>6} chars  ≈ {total_tokens:>5} tokens")

---

## 7. End-to-End Answer Comparison

Send the same query to the LLM with baseline context vs. enriched context.
For questions that span chunk boundaries, the enriched answer should be
more complete and accurate.

In [ ]:
from conversational_toolkit.utils.retriever import build_query_with_chunks
from sme_kt_zh_collaboration_rag.feature0_baseline_rag import SYSTEM_PROMPT


async def answer_with_context(query: str, chunks, label: str) -> str:
    context = build_query_with_chunks(query, list(chunks))
    from conversational_toolkit.llms.base import LLMMessage, Roles

    messages = [
        LLMMessage(role=Roles.SYSTEM, content=SYSTEM_PROMPT),
        LLMMessage(role=Roles.USER, content=context),
    ]
    response = await llm.send_message(messages)
    print(f"── {label} " + "─" * (60 - len(label)))
    print(response.content)
    print()
    return response.content


ANSWER_QUERY = (
    "What is the GWP of the product and how is it distributed across life cycle stages?"
)
print(f"Query: {ANSWER_QUERY!r}\n")

baseline_for_answer = await retrieve_baseline(
    ANSWER_QUERY, embedding_model, vector_store, top_k=3
)
enriched_for_answer = await retrieve_with_context_window(
    ANSWER_QUERY, embedding_model, vector_store, top_k=3, window_size=1
)

_ = await answer_with_context(
    ANSWER_QUERY, baseline_for_answer.chunks, "Baseline (no context expansion)"
)
_ = await answer_with_context(
    ANSWER_QUERY, enriched_for_answer.chunks, "Enriched (window_size=1)"
)

> **Observation:** For questions that span chunk boundaries (e.g. a numerical value
mentioned in one chunk and its context in the previous chunk), the enriched answer
should be more complete. If both answers are identical, the retrieval already found
self-contained chunks, which is common with header-based chunking but less so with
fixed-size chunking.

---

## Reference Queries: Context Enrichment Progress

Context expansion helps most when the answer spans chunk boundaries (e.g. a numerical value in one chunk and its unit or context in the next).

| Query | Baseline | Context window (w=1) |
|---|---|---|
| "What is the GWP of the Logypal 1 pallet?" | May retrieve only the table row with the value, missing the declaration header | Neighbouring chunks add the declaration context |
| "Is any product FSC-C147829 certified?" | Retrieves the FSC mention but not surrounding text explaining the certification scope | Window adds the certification scope from adjacent chunks |
| "What carbon neutrality claims does tesa make?" | May split the claim and its verification evidence across chunks | Window merges them into a single context |

---

## Summary

| Aspect | Baseline | Context window (w=1) | Context window (w=2) |
|---|---|---|---|
| Chunk size | ~800 chars | ~2 400 chars | ~4 000 chars |
| Token cost (5 chunks) | ~1 000 | ~3 000 | ~5 000 |
| Cross-boundary completeness | Low | High | Very high |
| Risk of irrelevant content | Low | Medium | Higher |

**When to use context enrichment:**
- Chunks from fixed-size or paragraph-aware chunking
- Questions that span multiple sections (GWP breakdown, full life cycle, multi-stage process)
- When baseline answers are incomplete or contain dangling references

**When to skip it:**
- Header-based chunks (already self-contained)
- Simple factual lookups where the answer fits in one chunk
- Very tight token budgets

**Combined recommended stack:**
hybrid retrieval (from `feature3_advanced_retrieval.ipynb`) + context window expansion
covers both vocabulary mismatch and context completeness with no extra LLM calls.